# Module 3.2 — SFT Data Prep for Instruction Fine-Tuning
**DeskMate SLM Curriculum · Phase 3 · Notebook 16**

Run on **CPU** — no GPU needed for data preparation.

By the end you will have:
- A tokenized SFT `DatasetDict` with loss-masked labels
- Bitext pairs + synthetic grounded replies merged and quality-filtered
- `data/processed/sft_dataset/` ready for Module 3.4 fine-tuning

Read `3.2_sft_dataprep.md` for full theory.

---

## Step 0 — Install & Seed

In [ ]:
%%capture
!pip install -q transformers==4.44.0 datasets==2.21.0 trl==0.10.1 rank-bm25==0.2.2 anthropic==0.34.0

In [ ]:
import random, json, pathlib, re, os, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datasets import Dataset, DatasetDict, load_dataset, load_from_disk
from transformers import AutoTokenizer
from trl import DataCollatorForCompletionOnlyLM
from torch.utils.data import DataLoader
from rank_bm25 import BM25Okapi

SEED = 42
random.seed(SEED); np.random.seed(SEED)
print('Libraries loaded.')

## Step 1 — Runtime & Paths

In [ ]:
try:
    import google.colab; RUNTIME = 'colab'
    PROJECT_ROOT = pathlib.Path('/content/slm')
except ImportError:
    try:
        import kaggle_secrets; RUNTIME = 'kaggle'
        PROJECT_ROOT = pathlib.Path('/kaggle/working/slm')
    except ImportError:
        RUNTIME = 'local'
        PROJECT_ROOT = pathlib.Path('.').resolve()

DATA_PROC  = PROJECT_ROOT / 'data' / 'processed'
DATA_PROC.mkdir(parents=True, exist_ok=True)
print(f'Runtime : {RUNTIME}')

## Step 2 — Load Tokenizer

In [ ]:
MODEL_NAME = 'Qwen/Qwen2.5-1.5B'
try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    print(f'Loaded tokenizer: {MODEL_NAME}')
except Exception as e:
    print(f'Could not load {MODEL_NAME}: {e}')
    print('Falling back to Qwen2.5-Instruct tokenizer (same vocab)')
    tokenizer = AutoTokenizer.from_pretrained(
        'Qwen/Qwen2.5-1.5B-Instruct', trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f'Vocab size      : {tokenizer.vocab_size:,}')
print(f'EOS token       : {tokenizer.eos_token!r} (id {tokenizer.eos_token_id})')
print(f'Chat template   : {"present" if tokenizer.chat_template else "not set"}')

In [ ]:
# Set a chat template if missing (base model may not have one)
DEFAULT_TEMPLATE = (
    '{%- for message in messages %}'
    '<|im_start|>{{ message["role"] }}\n{{ message["content"] }}<|im_end|>\n'
    '{%- endfor %}'
    '{%- if add_generation_prompt %}<|im_start|>assistant\n{%- endif %}'
)
if not tokenizer.chat_template:
    tokenizer.chat_template = DEFAULT_TEMPLATE
    print('Chat template set (base model default).')
else:
    print('Chat template already present.')

# Verify template works
test_msgs = [
    {'role': 'system', 'content': 'You are DeskMate.'},
    {'role': 'user', 'content': 'I cannot log in.'},
    {'role': 'assistant', 'content': 'I can help reset your password.'},
]
rendered = tokenizer.apply_chat_template(test_msgs, tokenize=False,
                                          add_generation_prompt=False)
print('\nRendered template example:')
print(rendered)

## Step 3 — Label Maps & Intents

In [ ]:
maps_path = DATA_PROC / 'label_maps.json'
if maps_path.exists():
    lm = json.loads(maps_path.read_text())
    INTENT2ID = lm['INTENT2ID']
    ID2INTENT  = {int(k): v for k, v in lm['ID2INTENT'].items()}
else:
    INTENTS   = ['account_access','account_settings','billing_dispute','billing_inquiry',
                 'cancellation','data_privacy','feature_request','onboarding',
                 'outage_report','payment_failure','performance_issue','refund_request',
                 'technical_bug','usage_question','out_of_scope']
    INTENT2ID = {i: n for n, i in enumerate(INTENTS)}
    ID2INTENT  = {n: i for n, i in enumerate(INTENTS)}

INTENTS = [i for i in INTENT2ID.keys() if i != 'out_of_scope']
print(f'SFT intents: {len(INTENTS)} (excluding out_of_scope)')

## Step 4 — Load Bitext Instruction/Response Pairs

Bitext pairs are high-quality human-written SFT examples.
We filter to intents that map into DeskMate's schema.

In [ ]:
BITEXT_INTENT_MAP = {
    'cancel_order':            'cancellation',
    'change_order':            'account_settings',
    'check_cancellation_fee':  'cancellation',
    'check_refund_policy':     'refund_request',
    'complaint':               'billing_dispute',
    'contact_customer_service':'usage_question',
    'contact_human_agent':     'usage_question',
    'create_account':          'onboarding',
    'delete_account':          'data_privacy',
    'edit_account':            'account_settings',
    'get_invoice':             'billing_inquiry',
    'get_refund':              'refund_request',
    'newsletter_subscription': 'feature_request',
    'payment_issue':           'payment_failure',
    'place_order':             'onboarding',
    'recover_password':        'account_access',
    'registration_problems':   'onboarding',
    'set_up_shipping_address': 'account_settings',
    'switch_account':          'account_settings',
    'track_order':             'usage_question',
    'track_refund':            'refund_request',
}

bitext_rows = []
try:
    raw_bt = load_dataset(
        'bitext/Bitext-customer-support-llm-chatbot-training-dataset', split='train')
    for ex in raw_bt:
        intent = BITEXT_INTENT_MAP.get(ex.get('intent',''), None)
        if intent and ex.get('instruction') and ex.get('response'):
            bitext_rows.append({
                'ticket':  ex['instruction'],
                'reply':   ex['response'],
                'intent':  intent,
                'context': None,
                'source':  'bitext',
            })
    print(f'Bitext: loaded {len(bitext_rows)} mapped examples')
except Exception as e:
    print(f'Bitext load failed ({e}) — using placeholder pairs')
    for intent in INTENTS:
        for i in range(8):
            bitext_rows.append({
                'ticket':  f'I need help with my {intent.replace("_"," ")} issue.',
                'reply':   f'Thank you for contacting us. We will resolve your '
                           f'{intent.replace("_"," ")} issue within 24 hours.',
                'intent':  intent, 'context': None, 'source': 'bitext_placeholder',
            })
    print(f'Placeholder: {len(bitext_rows)} rows')

bitext_df = pd.DataFrame(bitext_rows)
print(bitext_df['intent'].value_counts().head(10).to_string())

## Step 5 — Synthetic FAQ Corpus & BM25 Retrieval Stub

A 15-doc FAQ corpus (one per intent). BM25 retrieves the most relevant snippet
to condition synthetic replies — replaced by dense RAG in Phase 4.

In [ ]:
FAQ_CORPUS = {
    'account_access': (
        'To reset your password, click Forgot Password on the login page. '
        'A reset link is sent to your registered email within 2 minutes. '
        'Check your spam folder if you do not receive it.'),
    'account_settings': (
        'You can update your profile, notification preferences, and timezone '
        'from Settings > Account. Changes take effect immediately.'),
    'billing_dispute': (
        'If you were charged incorrectly, open a dispute from Billing > Transactions. '
        'Our team reviews disputes within 3 business days and issues a credit if valid.'),
    'billing_inquiry': (
        'Your current plan, billing cycle, and invoice history are available '
        'under Billing > Invoices. Enterprise pricing is available on request.'),
    'cancellation': (
        'To cancel your subscription, go to Settings > Billing > Cancel Plan. '
        'Your access continues until the end of the current billing period.'),
    'data_privacy': (
        'To request deletion of your data under GDPR or CCPA, email privacy@deskmate.io. '
        'Deletion is completed within 30 days. Account access is revoked immediately.'),
    'feature_request': (
        'We welcome feature requests. Submit them at feedback.deskmate.io '
        'or vote on existing requests. Popular requests are added to the roadmap quarterly.'),
    'onboarding': (
        'After signing up, connect your first data source under Integrations. '
        'The setup wizard guides you through each step. Setup takes about 10 minutes.'),
    'outage_report': (
        'Service status is available at status.deskmate.io. '
        'During an outage, our SRE team posts updates every 30 minutes. '
        'Subscribe to status alerts to be notified automatically.'),
    'payment_failure': (
        'Payment failures are usually caused by an expired card or insufficient funds. '
        'Update your payment method under Billing > Payment Methods. '
        'We retry failed payments after 3 days.'),
    'performance_issue': (
        'If the dashboard is slow, try clearing your browser cache or using Chrome. '
        'For persistent slowness, check status.deskmate.io and contact support '
        'with your account ID and the affected page URL.'),
    'refund_request': (
        'Refunds for annual plans are prorated for unused months. '
        'Monthly plan refunds are evaluated case by case. '
        'Submit a refund request from Billing > Request Refund.'),
    'technical_bug': (
        'To report a bug, go to Help > Report an Issue. Include your browser version, '
        'OS, and steps to reproduce. Our engineering team triages bugs within 24 hours.'),
    'usage_question': (
        'Full documentation is available at docs.deskmate.io. '
        'For video walkthroughs, visit our YouTube channel. '
        'Live chat support is available Monday-Friday 9am-6pm PST.'),
    'out_of_scope': (
        'DeskMate is a B2B support automation platform. '
        'Questions unrelated to our product cannot be answered here.'),
}

# Build BM25 index
corpus_docs    = list(FAQ_CORPUS.values())
corpus_intents = list(FAQ_CORPUS.keys())
corpus_tokens  = [doc.lower().split() for doc in corpus_docs]
bm25_index     = BM25Okapi(corpus_tokens)

def retrieve_snippet(query, top_k=1):
    scores = bm25_index.get_scores(query.lower().split())
    best   = int(np.argmax(scores))
    return corpus_docs[best]

# Test retrieval
test_q = 'I was charged twice on my credit card'
snippet = retrieve_snippet(test_q)
print(f'Query  : {test_q}')
print(f'Snippet: {snippet[:120]}...')

## Step 6 — Generate Synthetic Grounded Replies

For each ticket from Module 2.2, retrieve a snippet and generate a reply.
Auto-detects Anthropic API / mock.

In [ ]:
# Detect API path
SYNTH_API = 'mock'
anthropic_client = None
try:
    import anthropic as _anth
    _key = None
    if RUNTIME == 'colab':
        try:
            from google.colab import userdata
            _key = userdata.get('ANTHROPIC_API_KEY')
        except Exception:
            pass
    _key = _key or os.environ.get('ANTHROPIC_API_KEY')
    if _key:
        anthropic_client = _anth.Anthropic(api_key=_key)
        SYNTH_API = 'anthropic'
except ImportError:
    pass
print(f'Synthetic reply API: {SYNTH_API}')

In [ ]:
SYSTEM_PROMPT = (
    'You are DeskMate, a concise customer support assistant. '
    'Given the context and the support ticket, write a helpful reply in 2-4 sentences. '
    'Be specific, actionable, and professional. Do not repeat the ticket back.'
)

def generate_reply(ticket, context, intent, api=SYNTH_API):
    if api == 'anthropic':
        user_msg = 'Context: ' + context + '\n\nTicket: ' + ticket
        msg = anthropic_client.messages.create(
            model='claude-haiku-4-5-20251001',
            max_tokens=150,
            system=SYSTEM_MSG,
            messages=[{'role': 'user', 'content': user_msg}],
        )
        return msg.content[0].text.strip()
    else:  # mock
        templates = [
            'Thank you for contacting DeskMate. {action} Please let us know if you need further assistance.',
            'We understand your concern regarding {topic}. {action} Our team is here to help.',
            'I can help with your {topic} issue. {action} Feel free to reach out if anything is unclear.',
        ]
        actions = {
            'account_access':    'You can reset your password from the login page.',
            'billing_dispute':   'Please open a dispute from Billing > Transactions.',
            'outage_report':     'Check status.deskmate.io for real-time updates.',
            'technical_bug':     'Please submit a bug report via Help > Report an Issue.',
            'refund_request':    'Submit a refund request from Billing > Request Refund.',
        }
        action = actions.get(intent, 'Please visit docs.deskmate.io for guidance.')
        tmpl   = templates[hash(ticket) % len(templates)]
        return tmpl.format(action=action, topic=intent.replace('_', ' '))

In [ ]:
# Load tickets from Module 2.2 (or use placeholder)
aug_path = DATA_PROC / 'train_augmented.jsonl'
if aug_path.exists():
    ticket_df = pd.read_json(aug_path, lines=True)
    # Sample up to 20 per intent to keep runtime manageable
    ticket_df = (ticket_df.groupby('intent', group_keys=False)
                 .apply(lambda g: g.sample(min(20, len(g)), random_state=SEED)))
    print(f'Loaded {len(ticket_df)} tickets from train_augmented.jsonl')
else:
    print('train_augmented.jsonl not found — using placeholder tickets')
    rows = []
    for intent in INTENTS:
        for i in range(10):
            rows.append({'text': f'I have a {intent.replace("_"," ")} problem #{i}.',
                         'intent': intent})
    ticket_df = pd.DataFrame(rows)

# Generate synthetic replies
synth_rows = []
for _, row in ticket_df.iterrows():
    ticket  = row['text']
    intent  = row.get('intent', 'usage_question')
    snippet = retrieve_snippet(ticket)
    reply   = generate_reply(ticket, snippet, intent)
    synth_rows.append({
        'ticket':  ticket, 'reply': reply, 'intent': intent,
        'context': snippet, 'source': 'synthetic',
    })

synth_df = pd.DataFrame(synth_rows)
print(f'Generated {len(synth_df)} synthetic grounded replies')

## Step 7 — Quality Filters

In [ ]:
def quality_filter(row):
    reply = row['reply']
    if not isinstance(reply, str) or len(reply.strip()) == 0:
        return False, 'empty_reply'
    tokens = reply.split()
    if len(tokens) < 8:
        return False, 'too_short'
    if len(tokens) > 200:
        return False, 'too_long'
    low = reply.lower()
    if low.startswith('context:') or low.startswith('ticket:'):
        return False, 'prompt_leakage'
    return True, 'pass'

def apply_filters(df, label):
    results = df.apply(quality_filter, axis=1)
    reasons = [r[1] for r in results]
    passed  = [r[0] for r in results]
    from collections import Counter
    counts  = Counter(reasons)
    print(f'{label}: {sum(passed)}/{len(df)} passed')
    for reason, cnt in counts.items():
        if reason != 'pass':
            print(f'  dropped ({reason}): {cnt}')
    return df[passed].reset_index(drop=True)

bitext_clean = apply_filters(bitext_df,  'Bitext')
synth_clean  = apply_filters(synth_df,   'Synthetic')

## Step 8 — Merge, Deduplicate & Split

In [ ]:
all_df = pd.concat([bitext_clean, synth_clean], ignore_index=True)
all_df = all_df.sample(frac=1, random_state=SEED).reset_index(drop=True)

# Simple dedup on ticket text
before = len(all_df)
all_df = all_df.drop_duplicates(subset='ticket').reset_index(drop=True)
print(f'After dedup: {len(all_df)} examples ({before - len(all_df)} removed)')

# 90/10 train/val split
n_val  = max(50, int(len(all_df) * 0.1))
val_df = all_df.iloc[:n_val]
trn_df = all_df.iloc[n_val:]
print(f'Train: {len(trn_df)}  Val: {len(val_df)}')

raw_sft = DatasetDict({'train': Dataset.from_pandas(trn_df),
                       'val':   Dataset.from_pandas(val_df)})

## Step 9 — Apply Chat Template & Tokenize

In [ ]:
SYSTEM_MSG = (
    'You are DeskMate, a concise customer support assistant. '
    'Respond in 2-4 sentences. Be specific and actionable.'
)
MAX_LEN = 512

def format_example(example):
    ticket  = example['ticket']
    reply   = example['reply']
    context = example.get('context')
    user_content = (('Context: ' + context + '\n\n') if context else '') + 'Ticket: ' + ticket
    messages = [
        {'role': 'system',    'content': SYSTEM_MSG},
        {'role': 'user',      'content': user_content},
        {'role': 'assistant', 'content': reply},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False)

def tokenize_sft(example):
    text = format_example(example)
    enc  = tokenizer(text, truncation=True, max_length=MAX_LEN,
                     padding=False, return_tensors=None)
    enc['labels'] = list(enc['input_ids'])  # copy; collator masks prompts
    return enc

print('Applying chat template and tokenizing ...')
drop_cols = [c for c in raw_sft['train'].column_names
             if c not in ('input_ids','attention_mask','labels')]
tokenized_sft = raw_sft.map(
    tokenize_sft, batched=False,
    remove_columns=drop_cols,
    desc='Tokenizing SFT',
)
print('Done.')
for split, ds in tokenized_sft.items():
    print(f'  {split}: {len(ds):>5} examples')

## Step 10 — Token Length Analysis

In [ ]:
lengths = [len(ex['input_ids']) for ex in tokenized_sft['train']]
print(f'Token length stats (train):')
print(f'  Median : {int(np.median(lengths))}')
print(f'  p95    : {int(np.percentile(lengths, 95))}')
print(f'  Max    : {max(lengths)}')
print(f'  >= 512 : {sum(l >= MAX_LEN for l in lengths)} '
       f'({sum(l>=MAX_LEN for l in lengths)/len(lengths)*100:.1f}%)')

fig, ax = plt.subplots(figsize=(9, 3))
ax.hist(lengths, bins=40, color='#4C72B0', edgecolor='white')
ax.axvline(MAX_LEN, color='red', linestyle='--', label=f'max_len={MAX_LEN}')
ax.set_xlabel('Token count'); ax.set_ylabel('Examples')
ax.set_title('SFT Example Length Distribution')
ax.legend(); plt.tight_layout(); plt.show()

## Step 11 — Loss Mask Demo

Show how `DataCollatorForCompletionOnlyLM` sets prompt tokens to `-100`.

In [ ]:
RESPONSE_TEMPLATE = '<|im_start|>assistant\n'
collator = DataCollatorForCompletionOnlyLM(
    response_template=RESPONSE_TEMPLATE,
    tokenizer=tokenizer,
)

tokenized_sft.set_format('torch', columns=['input_ids','attention_mask','labels'])

sample = [tokenized_sft['train'][0]]
batch  = collator(sample)
ids    = batch['input_ids'][0].tolist()
lbls   = batch['labels'][0].tolist()
toks   = tokenizer.convert_ids_to_tokens(ids)

print('Loss mask (first 40 tokens of first example):')
print(f'{"Token":<20} {"Label"}')
print('-' * 40)
for tok, lbl in list(zip(toks, lbls))[:40]:
    lbl_str = str(lbl) if lbl != -100 else '-100 (IGNORED)'
    print(f'{tok:<20} {lbl_str}')

## Step 12 — DataLoader Smoke Test

In [ ]:
loader = DataLoader(tokenized_sft['train'], batch_size=4,
                    collate_fn=collator, shuffle=True)
batch  = next(iter(loader))
print('Batch shapes:')
for k, v in batch.items():
    print(f'  {k:<20}: {tuple(v.shape)}  dtype={v.dtype}')
n_masked   = (batch['labels'] == -100).sum().item()
n_unmasked = (batch['labels'] != -100).sum().item()
total      = batch['labels'].numel()
print(f'\nLabels: {n_unmasked} real ({n_unmasked/total*100:.0f}%) '
       f'/ {n_masked} masked ({n_masked/total*100:.0f}%)')

## Step 13 — Save SFT Dataset

In [ ]:
save_path = DATA_PROC / 'sft_dataset'
tokenized_sft.save_to_disk(str(save_path))
print(f'SFT dataset saved: {save_path}')

# Verify reload
from datasets import load_from_disk
check = load_from_disk(str(save_path))
print(f'Reload: {list(check.keys())} — train={len(check["train"])}, val={len(check["val"])}')
assert len(check['train']) == len(tokenized_sft['train'])
print('Reload verified.')

## Checkpoint

> **Why do we mask the prompt tokens from the loss?**

Write your answer below. Cover:
1. What the loss measures.
2. Why predicting prompt tokens is unhelpful.
3. What `-100` does in PyTorch's cross-entropy.
4. Observable difference if you forget to mask.

In [ ]:
answer = """
[Write your answer here]
"""
print(answer)

## Deliverable Summary

| Artifact | Location |
|---|---|
| SFT dataset | `data/processed/sft_dataset/` |

**What you've built:** a loss-masked, chat-templated SFT dataset combining
Bitext instruction pairs and BM25-grounded synthetic replies.

**Next:** Module 3.3 — PEFT theory: LoRA and QLoRA.
Understand what LoRA does to weight matrices, how rank `r` controls
the parameter budget, and how QLoRA fits on a free T4.